# Prepare representations, then score candidates

This notebook runs against the package bundled with this documentation. In a local kernel, install this checkout with `pip install -e .` first.

In [1]:
import sys

if sys.platform == "emscripten":
    import piplite

    await piplite.install("stagecoachml")

## One representation per user

Train an interaction model, encode two users, and score three candidates. Alice appears twice; her representation is computed once.

In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, PolynomialFeatures

from stagecoachml import StagecoachClassifier

rng = np.random.default_rng(7)
X = pd.DataFrame(rng.normal(size=(1000, 3)), columns=["interest", "item", "fresh"])
y = (X.interest * X.item + 0.3 * X.fresh > 0).astype(int)

model = StagecoachClassifier(
    encoder=FunctionTransformer(),
    estimator=make_pipeline(
        PolynomialFeatures(degree=2, include_bias=False),
        LogisticRegression(),
    ),
    upstream_features=["interest"],
    online_features=["item", "fresh"],
).fit(X, y)

users = pd.DataFrame({"interest": [-1.0, 1.0]})
state = model.encode(users, entity_ids=["alice", "bob"], feature_times=1000.0)

candidates = pd.DataFrame({"item": [-1.0, 1.0, 1.0], "fresh": [0.1, 0.1, 0.2]})
rows = state.rows_for(["alice", "alice", "bob"])
probabilities = model.predict_proba_from_state(
    candidates,
    state,
    state_indices=rows,
    as_of=1010.0,
    max_age=60.0,
)

probabilities

array([[1.01168104e-04, 9.99898832e-01],
       [9.99844397e-01, 1.55603344e-04],
       [7.08520940e-05, 9.99929148e-01]])

## Verify independent serving

The exported predictor contains no encoder. It consumes only online features and matching state.

In [3]:
from unittest.mock import patch

with patch.object(
    model.encoder_, "transform", side_effect=AssertionError("encoder executed")
):
    independent = model.predictor_.predict_proba(
        candidates, state, state_indices=rows, as_of=1010.0, max_age=60.0
    )
np.testing.assert_allclose(independent, probabilities)
print("Prepared scoring agrees and makes zero encoder calls.")

Prepared scoring agrees and makes zero encoder calls.


## Enforce freshness

The cutoff timestamp is the age of the input information. Expired state raises an error so the application can refresh it or select an explicit fallback.

In [4]:
try:
    model.predict_proba_from_state(
        candidates, state, state_indices=rows, as_of=1100.0, max_age=60.0
    )
except ValueError as error:
    print(error)

State exceeds max_age; refresh it before scoring
